In [ ]:
# import key libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)

# set working directory & load FULL data from local storage
setwd("C:/Users/jacob/water_current_MA/data")

# load the early late (first and last 2 trials/phase) dataset for dual target training
df_early_late_2_single <- read.csv("spatial_generalization/early_late_phase_2_PCA_error_novel_TU_3.0M_spatial_generalization_novel_TD_3.0M_spatial_generalization_speed_neg3_0.csv")

# load full dfs
df_single <- read.csv("spatial_generalization/PCA_error_novel_TU_3.0M_spatial_generalization_novel_TD_3.0M_spatial_generalization_speed_neg3_0_0_0.csv")

df_names <- ls(pattern = "df_")
df_list <- mget(df_names)
df_list

# loop through both df to adjust data types
for (i in seq_along(df_list)) {

    df <- df_list[[i]]

    # Format data types 
    df$ppid_full <- factor(df$ppid_full)
    df$speed_label <- factor(df$speed_label)
    df$target_x_label <- factor(df$target_x_label)
    df$phase <- factor(df$phase)
    
    df$set_order <- factor(df$set_order)
    # change set_order names to reflect thesis, group 1 and group 2 labels
    levels(df$set_order) <- c("group_1","group_2")
    
    # ensure consistent levels
    df$target_x_label <- factor(df$target_x_label, 
                                         levels = c("L60", "L30", "R30", "R60"))

    df_list[[i]] <- df
    }

list2env(df_list, envir = .GlobalEnv)




Warning message:
"package 'emmeans' was built under R version 4.4.3"
Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'afex' was built under R version 4.4.3"
Loading required package: lme4

Warning message:
"package 'lme4' was built under R version 4.4.3"
Loading required package: Matrix

************
Welcome to afex. For support visit: http://afex.singmann.science/

- Functions for ANOVAs: aov_car(), aov_ez(), and aov_4()
- Methods for calculating p-values with mixed(): 'S', 'KR', 'LRT', and 'PB'
- 'afex_aov' and 'mixed' objects can be passed to emmeans() for follow-up tests
- Get and set global package options with: afex_options()
- Set sum-to-zero contrasts globally: set_sum_contrasts()
- For exa

# Training Phase 1: early and late 2 trials

In [3]:
# Isolate training phase
df_early_late_2_single_t1 <- df_early_late_2_single[df_early_late_2_single$phase == 'training_1',]
speeds_list <- unique(df_early_late_2_single_t1$speed_label)
condition_list <- unique(df_early_late_2_single_t1$experiment)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (condition in condition_list) {
        
        df_subset <- df_early_late_2_single_t1[df_early_late_2_single_t1$speed_label == speed & df_early_late_2_single_t1$experiment == condition, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = "trial_set",
            fun_aggregate = mean 
        )
        
        label <- paste(speed, condition)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x CONDITION:", label, "\n")
                # cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                # if (p_adj_inter <= 0.05) {
                #     cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                # #     print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                # } else 
                  if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x CONDITION: -3 novel_TU_3.0M_spatial_generalization 
Holm-Adjusted trial_set Main Effect p-value:  3.208936e-09 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
     Effect    df    MSE          F  ges p.value
1 trial_set 1, 19 285.07 115.71 *** .591   <.001
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     57.4 5.34 19  10.757  <.0001


ANALYSIS FOR SPEED x CONDITION: -3 novel_TD_3.0M_spatial_generalization 
Holm-Adjusted trial_set Main Effect p-value:  6.663544e-06 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
     Effect    df    MSE         F  ges p.value
1 trial_set 1, 15 186.66 45.40 *** .596   <.001
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     32.5 4.83 

Because both conditions show a main effect of trial_set duringn T1, this provides clear evidence of learning. 

# Training Phase 2: early and late 2 trials

In [6]:
# Isolate training phase
df_early_late_2_single_t2 <- df_early_late_2_single[df_early_late_2_single$phase == 'training_2',]
speeds_list <- unique(df_early_late_2_single_t2$speed_label)
condition_list <- unique(df_early_late_2_single_t2$experiment)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (condition in condition_list) {
        
        df_subset <- df_early_late_2_single_t2[df_early_late_2_single_t2$speed_label == speed & df_early_late_2_single_t2$experiment == condition, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = "trial_set",
            fun_aggregate = mean 
        )
        
        label <- paste(speed, condition)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x CONDITION:", label, "\n")
                # cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                # if (p_adj_inter <= 0.05) {
                #     cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                # #     print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                # } else 
                  if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x CONDITION: -3 novel_TU_3.0M_spatial_generalization 
Holm-Adjusted trial_set Main Effect p-value:  0.248342 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
     Effect    df    MSE    F  ges p.value
1 trial_set 1, 19 417.30 1.42 .048    .248
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Neither effect survived Holm correction. Skipping post-hocs.

ANALYSIS FOR SPEED x CONDITION: -3 novel_TD_3.0M_spatial_generalization 
Holm-Adjusted trial_set Main Effect p-value:  2.018934e-05 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
     Effect    df    MSE         F  ges p.value
1 trial_set 1, 15 258.50 42.20 *** .480   <.001
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     36.9 5.68 15   6.496  <.0001



This result is in contrast with the T1 RM ANOVA, where we can only see a significant main effect of trial_set for the condition who trained downstream in T1, and transfered to an upstream target during T2

# Between-Subjects Transfer Test

In [9]:

# Filter for Transfer (Early trials only)
df_transfer <- df_early_late_2_single[df_early_late_2_single$trial_set == 'early',]
df_transfer$phase <- factor(df_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_transfer$speed_label)
target_list <- unique(df_transfer$target_x_label)



for (s in speeds_list) {

    p_vals <- c()
    
    for (t in target_list) {
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_transfer[df_transfer$speed_label == s & df_transfer$target_x_label == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset, alternative = "greater")
          print(res)

          # show sd
          print('sd')
          print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$phase, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'experiment:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'experiment:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}



--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 R60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 2.5364, df = 68.497, p-value = 0.006742
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 5.351725      Inf
sample estimates:
mean in group training_1 mean in group training_2 
                76.51333                 60.89298 

[1] "sd"
training_1 training_2 
   27.0847    25.0356 

Cohen's d

d estimate: 0.5962642 (medium)
95 percent confidence interval:
    lower     upper 
0.1129723 1.0795562 


--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 5.895, df = 69.464, p-value = 6.121e-08
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 25.5586     Inf
samp

While initial errors are significantly lower in Training Phase 2 (T2) for both targets, near complete transfer was observed for Target L60 in T2. This suggests that prior experience with R60 during T1 robustly transfered to L60, while a smaller but significant initial error reduction was observed for R60 in T2. Given the task structure, perhaps training with the constrained, R60 solution manifold allows participants to interpolate the novel manifold when transfering to L60, whereas the opposite target presentation order requires extrapolation. Furthermore, launching to R60 requires participants to not only control for unpredicitable water-ball dynamics, but to launch against the water current. While launching to L60 may primarily involve predicting how the ball will be carried downstream, and does not require launching against the current, meaning the perturbation is more forgiving for this target. 